# Hola, QuantLab

Bienvenido al Torneo de Ciencia de Datos ML de QuantLab.

Este notebook está diseñado para ayudarte a construir tu primer modelo de machine learning y empezar a competir en el torneo.

En este notebook vamos a:
1. Descargar y explorar el dataset de QuantLab
2. Entrenar tu primer modelo de machine learning
3. Evaluarlo con correlación por era y consistencia
4. Generar `predictions.csv` para enviarlo al torneo

> QuantLab es una herramienta de investigación. No es asesoría financiera ni recomendación de inversión.

In [ ]:
!python --version
!pip install -q pandas pyarrow scikit-learn requests matplotlib

## 1. Dataset

El dataset de QuantLab es un panel tabular que describe el mercado a lo largo del tiempo, compilado de datos que serían difíciles de obtener individualmente.

También está **ofuscado**: los ids de activos, los nombres de features y la definición del target están anonimizados. Esto permite que QuantLab reparta los datos gratis y que los modeles se construyan sin conocimiento de dominio financiero (¡ni sesgo!).

A diferencia de los torneos de código tradicionales, aquí **subes predicciones** (`id, prediction`) sobre un dataset, no código. El servidor verifica tu estrategia por replicabilidad y la compara contra benchmarks.

In [ ]:
import os
import io
import requests
import pandas as pd

# El worker de QuantLab expone todo vía HTTP. Usamos la URL pública del worker.
# (Es la misma que usa el frontend; no necesitas token para listar datasets.)
WORKER_URL = os.environ.get("QUANTLAB_WORKER_URL", "https://quantlab2.onrender.com").rstrip("/")

# El torneo ML es único (slug 'ml-sintetico'); lo obtenemos automáticamente.
r = requests.get(f"{WORKER_URL}/tournament/ml", timeout=30)
r.raise_for_status()
torneos = r.json()
assert torneos, "No hay torneo ML disponible. El worker puede estar generando la ronda."
TORNEO = torneos[0]
TOURNAMENT_ID = TORNEO["id"]
print("Torneo:", TORNEO["name"])
print("Ronda actual:", TORNEO.get("round_number"))
print("Cierra:", TORNEO.get("submission_deadline"))

### Listando los datasets de la ronda

Cada ronda tiene tres archivos:
- `train`: para entrenar tu modelo.
- `validation`: para validar fuera de muestra (out-of-sample).
- `live`: el holdout real. **Nunca se expone su parquet**; solo envías predicciones sobre él.

In [ ]:
def listar_datasets(tournament_id, round_number=None):
    params = {}
    if tournament_id:
        params["tournament_id"] = tournament_id
    if round_number is not None:
        params["round_number"] = round_number
    r = requests.get(f"{WORKER_URL}/ml/datasets", params=params, timeout=60)
    r.raise_for_status()
    return r.json().get("datasets", [])

def url_descarga(dataset):
    if dataset.get("download_url"):
        return dataset["download_url"]
    r = requests.get(
        f"{WORKER_URL}/ml/datasets/{dataset['id']}/download",
        params={"kind": dataset["kind"]},
        timeout=60,
    )
    r.raise_for_status()
    return r.json()["url"]

def leer_parquet(url):
    r = requests.get(url, timeout=300)
    r.raise_for_status()
    return pd.read_parquet(io.BytesIO(r.content))

datasets = listar_datasets(TOURNAMENT_ID)
for d in datasets:
    print(
        f"ronda {d['round_number']:>3} | {d['kind']:<10} | modo={d['mode']:<9} "
        f"| estado={d['status']:<8} | filas={d.get('row_count')} "
        f"| activos={d.get('n_assets')} eras={d.get('n_eras')} features={d.get('n_features')}"
    )
if not datasets:
    print("Sin datasets: el worker aún está generando la ronda.")

### Descargando train / validation

Usamos la URL pública de cada dataset (el mismo endpoint que el botón **Descargar** de la web).

In [ ]:
por_kind = {}
for d in sorted(datasets, key=lambda x: x.get("round_number") or 0):
    por_kind[d["kind"]] = d  # queda la ronda más alta de cada kind

ds_train = por_kind.get("train")
ds_valid = por_kind.get("validation")
ds_live = por_kind.get("live")
assert ds_train is not None, "La ronda no tiene dataset de train."

train = leer_parquet(url_descarga(ds_train))
valid = leer_parquet(url_descarga(ds_valid)) if ds_valid else None

print("train:", train.shape)
print("validation:", None if valid is None else valid.shape)
train.head()

### Filas, eras y target

Cada fila representa un activo en un momento del tiempo:
- `id` es el id del activo (ofuscado).
- `era` es el periodo temporal (las eras están ~1 día o 1 semana separadas, según el timeframe).
- `target` es una medida de retorno futuro de ese activo, ya regularizada en bins.
- `feature_*` describen atributos del activo para esa era.

**Era:** piensa en las filas dentro de una `era` como un solo ejemplo de mercado. A lo largo del notebook medimos todo *por era* — la correlación promedio por era es la métrica del torneo, no una sola cifra sobre todo el set.

In [ ]:
import matplotlib.pyplot as plt

# Tamaño de muestra por era
train.groupby("era").size().plot(
    title="Filas por era (train)", figsize=(5, 3), xlabel="era"
)
plt.show()

# Distribución del target
train["target"].plot(
    kind="hist", title="Target (train)", figsize=(5, 3),
    xlabel="valor", density=True, bins=50,
)
plt.show()

### Features

Las `feature_*` son atributos cuantitativos de cada activo. Sus definiciones no importan: QuantLab las incluye porque ayudan a predecir el `target`. No inventes columnas: usa las que declara el dataset (`feature_cols`).

In [ ]:
features = ds_train.get("feature_cols") or [c for c in train.columns if c.startswith("feature_")]
features = [c for c in features if c in train.columns]
assert features, "No se encontraron columnas feature_*."

TARGET = "target"
assert TARGET in train.columns, f"El train no tiene '{TARGET}'."

print(f"{len(features)} features. Ejemplo: {features[:5]}")

X = train[features].astype("float32").fillna(0.5)
y = train[TARGET].astype("float32")
mask = y.notna()
X, y = X[mask], y[mask]
print("entrenamiento:", X.shape)

## 2. Modeling

Nuestra tarea: predecir el `target` usando las `features`.

### Entrenamiento

Eres libre de usar cualquier framework, pero aquí usamos un modelo pequeño de scikit-learn. El objetivo es un baseline honesto, no ganar la ronda.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

modelo = GradientBoostingRegressor(
    n_estimators=120, max_depth=3, learning_rate=0.05, subsample=0.8, random_state=42
)
modelo.fit(X, y)
print("entrenado.")

### Validación (correlación por era)

La métrica del torneo es la correlación de Spearman media por era, así que se mide igual aquí. La consistencia (fracción de eras con correlación positiva) importa tanto como la correlación promedio.

In [ ]:
import numpy as np

def corr_por_era(df, col_pred, col_target=TARGET):
    col_era = "era" if "era" in df.columns else None
    if col_era is None:
        return pd.Series({"global": df[col_pred].corr(df[col_target], method="spearman")})
    return df.groupby(col_era).apply(
        lambda g: g[col_pred].corr(g[col_target], method="spearman")
    )

if valid is not None and TARGET in valid.columns:
    Xv = valid[features].astype("float32").fillna(0.5)
    valid = valid.copy()
    valid["prediction"] = modelo.predict(Xv)
    corrs = corr_por_era(valid.dropna(subset=[TARGET]), "prediction")
    print(f"corr media   : {corrs.mean():.4f}")
    print(f"desv. std    : {corrs.std():.4f}")
    print(f"sharpe corr  : {corrs.mean() / corrs.std():.3f}" if corrs.std() else "")
    print(f"consistencia : {(corrs > 0).mean():.2%} de eras positivas")
else:
    print("Sin validation con target: se omite la validación.")

### Por qué importa la correlación por era

Un modelo puede tener correlación alta en promedio pero negativa en muchas eras (overfitting a eras específicas). La consistencia premia modelos que aciertan *en la mayoría* de los periodos, no solo en promedio. Es la misma idea que el *Sharpe* en finanzas: retorno dividido por riesgo.

Las puntuaciones son bajas: +/- 0.05 es normal en finanzas cuantitativas. Por eso QuantLab verifica la **replicabilidad** de tu estrategia y la compara contra benchmarks (Buy & Hold, Media Móvil) en el Sello de Integridad.

## 3. Generar `predictions.csv`

El dataset `live` no se descarga (nunca se expone su parquet). Predices sobre `validation` para dejar el CSV con el formato exacto; cuando la ronda `live` esté abierta, usa el mismo código sobre las filas que te entregue la ronda.

**Formato obligatorio:** exactamente dos columnas, `id` y `prediction`, sin nulos y numéricas.

In [ ]:
base = valid if valid is not None else train
Xb = base[features].astype("float32").fillna(0.5)

col_id = "id" if "id" in base.columns else base.columns[0]
salida = pd.DataFrame({
    "id": base[col_id].astype(str).values,
    "prediction": modelo.predict(Xb).astype("float64"),
})

# Validación local idéntica a la del worker
assert list(salida.columns) == ["id", "prediction"], salida.columns
assert not salida["prediction"].isna().any(), "hay NaN en prediction"
assert len(salida) > 0, "CSV vacío"

salida.to_csv("predictions.csv", index=False)
print(f"predictions.csv escrito: {len(salida)} filas")
salida.head()

## 4. Enviar

Arrastra `predictions.csv` a la pestaña **Enviar** del torneo en QuantLab (web). El envío reemplaza el anterior de la misma ronda, así que puedes iterar sin penalización.

Ideas para mejorar el baseline: neutralizar features, promediar varias semillas, validar por eras contiguas (no aleatorias) y vigilar la consistencia además de la correlación.

> Recuerda: el servidor verifica la replicabilidad de tu estrategia y la compara contra benchmarks. Una buena correlación por era y alta consistencia es lo que el Sello de Integridad premia.